# Ontology Quality Suite -- interactive primer

A runnable companion to `docs/PRIMER.md` -- every code cell below actually
executes against this repo's own `examples/` fixtures (run this notebook
from the repo root). Markdown cells stay short and link back to
`docs/PRIMER.md` and the other `docs/*.md` files for full narrative depth;
this notebook's job is to *show it actually working*, not replace the prose.

Sections mirror `docs/PRIMER.md`'s use cases, skipping the two that can't
meaningfully execute in a notebook (CI YAML, the maturity-path narrative) --
see that doc directly for those.

New to RDF/OWL? `docs/PRIMER.md` opens with a short, plain-language glossary
(ontology/TBox, data graph/ABox, triple, IRI, SHACL vs. SPARQL, reasoner,
taxonomy, blank node) -- worth a skim before this notebook if any of those
are unfamiliar.

## 0. Setup

In [1]:
import subprocess, sys, json as _json, tempfile, shutil, os
from pathlib import Path
import pandas as pd

def _find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("could not find the repo root (looked for pyproject.toml + examples/) above " + str(start))

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
print("repo root:", REPO_ROOT)

def run(*args, cwd=REPO_ROOT, check=True):
    """Runs the ontology-quality-suite CLI and prints its stdout. Nothing in
    this notebook needs the exit code/CompletedProcess itself, so this
    returns nothing -- keeps Jupyter's REPL-style auto-display from dumping
    a raw CompletedProcess repr under every call."""
    proc = subprocess.run(["ontology-quality-suite", *args], cwd=cwd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0 and proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if check and proc.returncode not in (0, 1):  # 1 = findings triggered --fail-on, still a clean run
        raise RuntimeError(f"exit {proc.returncode}")

def summary(out_dir):
    """Reads full_results.csv and returns a per-check-id finding count Series."""
    df = pd.read_csv(Path(out_dir) / "full_results.csv")
    return df.groupby(["check_id", "severity"]).size()

run("--help")

repo root: C:\repos\consolidated_ontology_suite_python


usage: ontology-quality-suite [-h]
                              {ontology,checks,sketch,triplify,data,docgen,run,version-diff,consistency,consistency-remote,pattern-consistency}
                              ...

Command-line interface for the consolidated ontology suite.

Subcommands mirror the pipeline stages in ``pipeline.py``:

    ontology-quality-suite ontology            --ontology domain.ttl
    ontology-quality-suite checks              --ontology domain.ttl [--data data.ttl]
    ontology-quality-suite sketch              --queries queries/ [--ontology domain.ttl]
    ontology-quality-suite triplify            --csv-dir csv/ --queries queries/
    ontology-quality-suite data                data.ttl [more.ttl ...] [--ontology domain.ttl]
    ontology-quality-suite docgen              --ontology domain.ttl [--instances data.ttl] [--ref imported.ttl ...]
    ontology-quality-suite run                 whichever of --ontology/--queries/--csv-dir/--data apply
    ontology-quality-s

## 1. Auditing an ontology you're actively authoring

See `docs/PRIMER.md` §4, `docs/REASONING.md`.

In [2]:
run("ontology", "--ontology", "examples/ontology/domain.ttl", "--out-dir", "out/nb-ontology")
summary("out/nb-ontology")

Findings: 2 total (1 Violation, 0 Warning, 1 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-ontology
  - out\nb-ontology\report.html (start here)
  - out\nb-ontology\full_results.csv



* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/Pedro/AppData/Local/Temp/tmpolt14lec -Y



check_id  severity 
LOG-001   Violation    1
REA-022   Info         1
dtype: int64

In [3]:
# Full 50-check registry too
run("checks", "--ontology", "examples/ontology/domain.ttl", "--out-dir", "out/nb-checks")
summary("out/nb-checks")

Findings: 31 total (2 Violation, 25 Warning, 4 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-checks
  - out\nb-checks\report.html (start here)
  - out\nb-checks\full_results.csv



check_id  severity 
EFF-001   Warning      2
LOG-001   Violation    2
LOG-003   Warning      2
QUA-001   Warning      8
QUA-002   Info         1
QUA-004   Warning      9
QUA-007   Info         1
STR-003   Warning      2
STY-001   Warning      1
STY-002   Warning      1
STY-003   Info         2
dtype: int64

In [4]:
# OWL2 profile membership -- off by default, ask explicitly
run("ontology", "--ontology", "examples/ontology/domain.ttl", "--profile", "EL", "--out-dir", "out/nb-el")
summary("out/nb-el")

Findings: 4 total (1 Violation, 0 Warning, 3 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-el
  - out\nb-el\report.html (start here)
  - out\nb-el\full_results.csv



* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/Pedro/AppData/Local/Temp/tmp6w8ij6hb -Y



check_id  severity 
LOG-001   Violation    1
REA-010   Info         2
REA-022   Info         1
dtype: int64

A real OWL2 DL reasoner's opinion, not just the always-on RDFS/OWL2-RL
closure. **Uses `--reasoner pellet`, not `hermit`**:
`examples/ontology/domain.ttl` declares a property with `rdfs:range
xsd:date`, which isn't in OWL2's official datatype map -- HermiT throws
`UnsupportedDatatypeException` on it and the suite degrades gracefully to
`REA-022` ("reasoner unavailable") rather than actually running. Pellet has
no such restriction. Needs `uv sync --extra reasoner` (owlready2 + a Java
runtime) -- skipped gracefully below if that's not set up here.

In [5]:
import importlib.util, shutil as _sh
has_reasoner = importlib.util.find_spec("owlready2") is not None and _sh.which("java") is not None
if has_reasoner:
    run("ontology", "--ontology", "examples/ontology/domain.ttl", "--reasoner", "pellet", "--out-dir", "out/nb-pellet")
    print(summary("out/nb-pellet"))
else:
    print("owlready2/Java not available in this environment -- skipping (see docs/REASONING.md)")

Findings: 3 total (3 Violation, 0 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-pellet
  - out\nb-pellet\report.html (start here)
  - out\nb-pellet\full_results.csv

check_id  severity 
LOG-001   Violation    1
REA-020   Violation    1
REA-021   Violation    1
dtype: int64


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\pellet\jena-core-fix

Python API, for scripting rather than the CLI:

In [6]:
from ontology_suite.ontologyeval import ontology_evaluation as oe
from ontology_suite.reasoning import profile

graph, import_report = oe.resolve_imports("examples/ontology/domain.ttl")
schema = oe.collect_schema(graph)
metrics = oe.compute_metrics(schema)
print({k: metrics[k] for k in list(metrics)[:8]})

profile_report = profile.check_profiles(graph, profiles=("EL",))
print("EL violations:", len(profile_report.violations) if hasattr(profile_report, "violations") else profile_report)

{'sizes': {'class_count': 12, 'anonymous_class_count': 0, 'property_count': 6, 'object_property_count': 2, 'datatype_property_count': 4, 'annotation_property_count': 0, 'subclass_edge_count': 11}, 'richness': {'attribute_richness': 0.333, 'relationship_richness': 0.154, 'inheritance_richness': 0.917}, 'instance_richness': None, 'anonymous_classes': [], 'completeness': {'annotated_class_count': 4, 'annotated_property_count': 6, 'domain_range_sources': {'domain_formal': 4, 'range_formal': 4}}, 'expressivity': {'equivalent_class_count': 1, 'disjoint_with_count': 1, 'equivalent_property_count': 0, 'inverse_of_count': 0, 'property_characteristics': {'FunctionalProperty': 1, 'InverseFunctionalProperty': 0, 'TransitiveProperty': 0, 'SymmetricProperty': 0, 'AsymmetricProperty': 0, 'ReflexiveProperty': 0, 'IrreflexiveProperty': 0}, 'restriction_count': 0, 'restriction_facet_counts': {}, 'restriction_subclass_count': 0}, 'per_class': {rdflib.term.URIRef('https://example.org/demo/L6'): {'instance

## 2. Building and validating a CSV-to-RDF pipeline

See `docs/PRIMER.md` §5, `docs/TARQL_ALIGNMENT.md`.

In [7]:
# 1. Static check: does the query reference vocabulary the ontology declares? No CSV needed.
run("sketch", "--queries", "examples/queries", "--ontology", "examples/ontology/domain.ttl", "--out-dir", "out/nb-sketch")

Findings: 12 total (0 Violation, 1 Warning, 11 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-sketch
  - out\nb-sketch\report.html (start here)
  - out\nb-sketch\full_results.csv
Sketch used 1 query file(s); 4 triples, 4 entities.



In [8]:
from ontology_suite import config
binary = config.find_oxi_gen_binary()
print("oxi-gen binary:", binary)
if binary:
    run("triplify", "--csv-dir", "examples/csv", "--queries", "examples/queries", "--out-dir", "out/nb-data")
else:
    print("no oxi-gen binary found (build it in a sibling ../oxi-gen checkout) -- skipping triplify+data below")

oxi-gen binary: C:\repos\oxi-gen\target\release\oxi_gen.exe


Triplified 1 file(s) into C:\repos\consolidated_ontology_suite_python\out\nb-data
  - out\nb-data\animals.ttl



In [9]:
if binary:
    run("data", "out/nb-data/animals.ttl", "--ontology", "examples/ontology/domain.ttl", "--out-dir", "out/nb-data-eval")
    print(summary("out/nb-data-eval"))

Findings: 58 total (9 Violation, 27 Warning, 22 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-data-eval
  - out\nb-data-eval\report.html (start here)
  - out\nb-data-eval\full_results.csv

check_id  severity 
CNF-002   Warning       1
CNF-005   Info         11
EFF-001   Warning       2
LOG-001   Violation     3
LOG-003   Warning       2
QUA-001   Warning       8
QUA-002   Info          1
QUA-004   Warning      10
QUA-007   Info          1
REA-022   Info          1
STR-002   Violation     3
STR-003   Warning       2
STR-007   Violation     3
STY-001   Warning       1
STY-002   Warning       1
STY-003   Info          8
dtype: int64


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit;C:\repos\consolidated_ontology_suite_python\.venv\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/Pedro/AppData/Local/Temp/tmpccus46l6 -Y



`--sample` caps the (expensive) reasoning pass to a Concise Bounded
Description of N named subjects, while the (cheap) registry checks still
run over the whole graph:

In [10]:
run("data", "examples/data/legacy-export.ttl", "--ontology", "examples/ontology/domain.ttl",
    "--sample", "5000", "--reasoner", "none", "--out-dir", "out/nb-sample")

Findings: 55 total (8 Violation, 32 Warning, 15 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-sample
  - out\nb-sample\report.html (start here)
  - out\nb-sample\full_results.csv
reasoning pass sampled 4 named subjects (of 4 total) via a Concise Bounded Description each



## 3. Releasing a new ontology version

See `docs/PRIMER.md` §6, `docs/VERSIONING.md`.

In [11]:
run("version-diff",
    "examples/gist_versions_reference/gistCore14.0.0.ttl",
    "examples/gist_versions_reference/gistCore14.1.0.ttl",
    "--out-dir", "out/nb-vdiff", "--json")
bump = _json.loads((Path("out/nb-vdiff") / "diff.json").read_text())["bump"]
print("detected bump:", bump)

Ontology version diff: examples/gist_versions_reference/gistCore14.0.0.ttl -> examples/gist_versions_reference/gistCore14.1.0.ttl

Added properties [minor]:
  - https://w3id.org/semanticarts/ns/ontology/gist/prohibits

Added equivalentProperty axioms [minor]:
  - https://w3id.org/semanticarts/ns/ontology/gist/prevents equivalentProperty https://w3id.org/semanticarts/ns/ontology/gist/prohibits

Suggested version bump: MINOR

Written to: out\nb-vdiff\diff.txt
Written to: out\nb-vdiff\diff.json

detected bump: minor


Before promoting a MAJOR change: run representative real data through the
*proposed* new version (the pre-deployment compatibility gate):

In [12]:
run("data", "examples/data/legacy-export.ttl", "--ontology", "examples/ontology/domain.ttl",
    "--sample", "5000", "--out-dir", "out/nb-predeploy", "--fail-on", "Violation", "--reasoner", "none")

Findings: 55 total (8 Violation, 32 Warning, 15 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-predeploy
  - out\nb-predeploy\report.html (start here)
  - out\nb-predeploy\full_results.csv
reasoning pass sampled 4 named subjects (of 4 total) via a Concise Bounded Description each



## 4. Rename detection and automated repair

See `docs/PRIMER.md` §8, `docs/CONSISTENCY_AND_REPAIR.md`. No committed
fixture in `examples/` demonstrates a deliberate rename with a migration
annotation, so this builds one inline, from scratch, in a temp directory --
a real, runnable scenario rather than an illustrative-only one.

In [13]:
scratch = Path(tempfile.mkdtemp(prefix="oq-primer-"))

old_ttl = scratch / "domain-v1.ttl"
old_ttl.write_text("""
@prefix ex: <https://example.org/rename-demo/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
ex:Widget a owl:Class .
ex:price a owl:DatatypeProperty ; rdfs:domain ex:Widget .
""", encoding="utf-8")

new_ttl = scratch / "domain-v2.ttl"
new_ttl.write_text("""
@prefix ex: <https://example.org/rename-demo/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
ex:Product a owl:Class .
ex:Widget dcterms:isReplacedBy ex:Product .
ex:cost a owl:DatatypeProperty ; rdfs:domain ex:Product .
ex:price dcterms:isReplacedBy ex:cost .
""", encoding="utf-8")

queries_dir = scratch / "queries"
queries_dir.mkdir()
(queries_dir / "transform.rq").write_text("""
PREFIX ex: <https://example.org/rename-demo/>
CONSTRUCT { ?w a ex:Widget ; ex:price \"1.00\" . }
WHERE { BIND(IRI(CONCAT(\"https://example.org/rename-demo/item-\", \"1\")) AS ?w) }
""", encoding="utf-8")
print("scratch dir:", scratch)

scratch dir: C:\Users\Pedro\AppData\Local\Temp\oq-primer-l3xpe6v5


In [14]:
from ontology_suite import consistency

report = consistency.check_consistency(
    str(new_ttl), old_ontology=str(old_ttl), tarql_sources=[str(queries_dir)],
)
print(consistency.format_consistency_report(report))
for r in report.repairs:
    print(f"- [{r.kind}] {r.target_file} (confidence {r.confidence:.0%}): {r.description}")

Ontology version diff: C:\Users\Pedro\AppData\Local\Temp\oq-primer-l3xpe6v5\domain-v1.ttl -> C:\Users\Pedro\AppData\Local\Temp\oq-primer-l3xpe6v5\domain-v2.ttl

Removed classes [MAJOR]:
  - https://example.org/rename-demo/Widget

Removed properties [MAJOR]:
  - https://example.org/rename-demo/price

Added classes [minor]:
  - https://example.org/rename-demo/Product

Added properties [minor]:
  - https://example.org/rename-demo/cost

Suggested version bump: MAJOR

1 class(es) used in TARQL but not declared in the ontology set:
  [undeclared_class] https://example.org/rename-demo/Widget
1 propert(y/ies) used in TARQL but not declared in the ontology set:
  [undeclared_property] https://example.org/rename-demo/price

1 suggested repair(s):
  [rename_iri] C:\Users\Pedro\AppData\Local\Temp\oq-primer-l3xpe6v5\queries\transform.rq (confidence 100%)
    Update to the renamed ontology term(s): https://example.org/rename-demo/Widget -> https://example.org/rename-demo/Product; https://example.org

In [15]:
print("--- transform.rq before ---")
print((queries_dir / "transform.rq").read_text())

applied = consistency.apply_repairs(report.repairs, min_confidence=0.7)
print(f"applied {len(applied)}/{len(report.repairs)} repair(s)")

print("--- transform.rq after ---")
print((queries_dir / "transform.rq").read_text())

--- transform.rq before ---

PREFIX ex: <https://example.org/rename-demo/>
CONSTRUCT { ?w a ex:Widget ; ex:price "1.00" . }
WHERE { BIND(IRI(CONCAT("https://example.org/rename-demo/item-", "1")) AS ?w) }

applied 1/1 repair(s)
--- transform.rq after ---

PREFIX ex: <https://example.org/rename-demo/>
CONSTRUCT { ?w a ex:Product ; ex:cost "1.00" . }
WHERE { BIND(IRI(CONCAT("https://example.org/rename-demo/item-", "1")) AS ?w) }



## 5. Taxonomy governance: `pattern-consistency`

See `docs/PRIMER.md` §9, `docs/MODELLING_PATTERN_CONSISTENCY.md`. Real
fixtures this time -- `transform.rq` deliberately references a taxonomy
value (`ex:Gasoline`) that doesn't exist (the taxonomy uses `ex:Petrol`).

In [16]:
run("pattern-consistency",
    "--queries", "examples/pattern_consistency/transform.rq",
    "--ontology", "examples/pattern_consistency/ontology.ttl",
    "--taxonomy", "examples/pattern_consistency/taxonomy.ttl",
    "--out-dir", "out/nb-pattern-broken")

== taxonomy <-> transformation ==
  [undeclared_taxonomy_reference] https://example.org/vehicle-demo/Gasoline is used as the value of https://w3id.org/semanticarts/ns/ontology/gist/isCategorizedBy in the TARQL query sketch but is not declared as an individual anywhere in the given taxonomy set.

Written to: out\nb-pattern-broken\pattern-consistency.txt



In [17]:
# The fixed version comes back clean
run("pattern-consistency",
    "--queries", "examples/pattern_consistency/transform-fixed.rq",
    "--ontology", "examples/pattern_consistency/ontology.ttl",
    "--taxonomy", "examples/pattern_consistency/taxonomy.ttl",
    "--out-dir", "out/nb-pattern-fixed")

No modelling-pattern inconsistencies found across ontology, taxonomy, transformation, or output data.

Written to: out\nb-pattern-fixed\pattern-consistency.txt



## 6. Running against a live triplestore

See `docs/PRIMER.md` §10, `docs/FUSEKI.md`. Needs a real SPARQL 1.1
Protocol endpoint -- this cell starts a small **real** in-process HTTP
server implementing just enough of the protocol to exercise the actual
`urllib`-based request/response code path (same technique
`tests/conftest.py` uses for this suite's own test suite; not a mock of
`urllib` itself), so the rest of this section is genuinely executable,
not just illustrative.

In [18]:
import threading, rdflib
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import parse_qs

_dataset = rdflib.Dataset()

class _Handler(BaseHTTPRequestHandler):
    def log_message(self, format, *args): pass
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        params = parse_qs(self.rfile.read(length).decode("utf-8"))
        if "update" in params:
            _dataset.update(params["update"][0])
            self.send_response(200); self.end_headers(); return
        query_text = params["query"][0]
        default_graphs = params.get("default-graph-uri", [])
        if default_graphs:
            merged = rdflib.Graph()
            for g in default_graphs:
                merged += _dataset.get_context(rdflib.URIRef(g))
            result = merged.query(query_text)
        else:
            result = _dataset.query(query_text)
        if result.type in ("CONSTRUCT", "DESCRIBE"):
            body = result.graph.serialize(format="turtle").encode("utf-8")
            self.send_response(200); self.send_header("Content-Type", "text/turtle"); self.end_headers()
            self.wfile.write(body)
        else:
            payload = {"head": {"vars": [str(v) for v in (result.vars or [])]}, "results": {"bindings": []}}
            body = _json.dumps(payload).encode("utf-8")
            self.send_response(200); self.send_header("Content-Type", "application/sparql-results+json"); self.end_headers()
            self.wfile.write(body)

_server = HTTPServer(("127.0.0.1", 0), _Handler)
_port = _server.server_address[1]
threading.Thread(target=_server.serve_forever, daemon=True).start()
QUERY_URL = f"http://127.0.0.1:{_port}/query"
print("fake SPARQL endpoint listening at", QUERY_URL)

fake SPARQL endpoint listening at http://127.0.0.1:49742/query


In [19]:
ONTOLOGY_1 = "https://example.org/graph/ontology/1.0.0"
ONTOLOGY_2 = "https://example.org/graph/ontology/2.0.0"

_dataset.get_context(rdflib.URIRef(ONTOLOGY_1)).parse(data=old_ttl.read_text(), format="turtle", publicID=ONTOLOGY_1)
_dataset.get_context(rdflib.URIRef(ONTOLOGY_2)).parse(data=new_ttl.read_text(), format="turtle", publicID=ONTOLOGY_2)

from ontology_suite.remote import fuseki
from ontology_suite.versioning import diff as version_diff

remote_dataset = fuseki.FusekiDataset(query_endpoint=QUERY_URL)
g1 = fuseki.load_named_graph(remote_dataset, ONTOLOGY_1)
g2 = fuseki.load_named_graph(remote_dataset, ONTOLOGY_2)
diff, bump = version_diff.diff_ontologies(g1, g2)
print("remote version-diff bump:", bump)

remote version-diff bump: BumpLevel.MAJOR


In [20]:
from ontology_suite.remote.manifest import GraphManifest, GraphBinding, check_manifest_consistency

DATA_GRAPH = "https://example.org/graph/triplified/widgets"
data_graph = _dataset.get_context(rdflib.URIRef(DATA_GRAPH))
data_graph.add((rdflib.URIRef("https://example.org/rename-demo/item-1"), rdflib.RDF.type, rdflib.URIRef("https://example.org/rename-demo/Widget")))

manifest = GraphManifest(bindings=[
    GraphBinding(graph_uri=ONTOLOGY_1, role="ontology"),
    GraphBinding(
        graph_uri=DATA_GRAPH, role="triplified_data",
        source_tarql=str(queries_dir / "transform.rq"), ontology_graph_uri=ONTOLOGY_1,
    ),
])
reports = check_manifest_consistency(remote_dataset, manifest)
for r in reports:
    print(r.graph_uri, "is_clean:", r.is_clean, "warnings:", r.warnings)

_server.shutdown()
print("fake endpoint stopped")

https://example.org/graph/triplified/widgets is_clean: False warnings: []


fake endpoint stopped


## 7. Reference documentation for stakeholders

See `docs/PRIMER.md` §11, `docs/CLASS_DIAGRAMS.md`.

In [21]:
run("docgen", "--ontology", "examples/ontology/domain.ttl", "--out-dir", "out/nb-docgen")
html = Path("out/nb-docgen/ontology-documentation.html")
print(html, html.stat().st_size, "bytes")
print(sorted(p.name for p in (Path("out/nb-docgen") / "class-diagrams").glob("*.svg"))[:5], "...")

Wrote out\nb-docgen\ontology_doc_data.json: prefix='ex', 12 classes, 2 object properties, 4 datatype properties, 1 sections, 0 external terms (0 resolved).
Wrote out\nb-docgen\ontology-documentation.html (12 classes, 2 object properties, 4 datatype properties, 0 external terms).
Reference documentation written to: out\nb-docgen\ontology-documentation.html
12 class diagram(s) written to: out\nb-docgen\class-diagrams

out\nb-docgen\ontology-documentation.html 63336 bytes
['Animal.svg', 'Feline.svg', 'L1.svg', 'L2.svg', 'L3.svg'] ...


## 8. Where the checks live, and customizing them without a local checkout

See `docs/PRIMER.md` §13, `docs/EXTENDING.md`. Confirms
`config.PACKAGE_RESOURCES` and the `--registry`/`--shapes`/`--sparql`
override mechanism live, in this environment, by actually removing a
check and re-running.

In [22]:
from ontology_suite import config
print("PACKAGE_RESOURCES:", config.PACKAGE_RESOURCES)

my_checks = scratch / "my-checks"
shutil.copytree(config.PACKAGE_RESOURCES, my_checks)
(my_checks / "sparql" / "quality" / "QUA-004.rq").unlink()
print("copied resources to", my_checks, "and removed QUA-004.rq")

PACKAGE_RESOURCES: C:\repos\consolidated_ontology_suite_python\ontology_suite\resources
copied resources to C:\Users\Pedro\AppData\Local\Temp\oq-primer-l3xpe6v5\my-checks and removed QUA-004.rq


In [23]:
run("checks", "--ontology", "examples/ontology/domain.ttl", "--engine", "sparql",
    "--out-dir", "out/nb-default-checks")
default_counts = summary("out/nb-default-checks")
print("QUA-004 findings (default resources):", default_counts.filter(like="QUA-004", axis=0).sum())

run("checks", "--ontology", "examples/ontology/domain.ttl", "--engine", "sparql",
    "--registry", str(my_checks / "registry.json"), "--shapes", str(my_checks / "shapes"), "--sparql", str(my_checks / "sparql"),
    "--out-dir", "out/nb-custom-checks")
custom_counts = summary("out/nb-custom-checks")
print("QUA-004 findings (custom resources, file removed):", custom_counts.filter(like="QUA-004", axis=0).sum())

Findings: 28 total (1 Violation, 24 Warning, 3 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-default-checks
  - out\nb-default-checks\report.html (start here)
  - out\nb-default-checks\full_results.csv

QUA-004 findings (default resources): 9


Findings: 19 total (1 Violation, 15 Warning, 3 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb-custom-checks
  - out\nb-custom-checks\report.html (start here)
  - out\nb-custom-checks\full_results.csv

QUA-004 findings (custom resources, file removed): 0


## Cleanup

In [24]:
shutil.rmtree(scratch, ignore_errors=True)
print("scratch dir removed")

scratch dir removed


## What's not in this notebook

- **CI wiring** (a GitHub Actions YAML template) -- `docs/PRIMER.md` §12.
- **The full ontology/taxonomy/data governance playbook** (roles, sequencing,
  rollback strategy) -- `docs/UPDATING.md`.
- **The maturity path for adopting this suite from scratch** --
  `docs/PRIMER.md` §14.
- **`--engine` (native Rust vs. pyshacl vs. portable SPARQL)** in depth,
  including the confirmed pyshacl severity-fidelity gap --
  `docs/ARCHITECTURE.md`.